# New Code!

This .ipynb will use the new application library rather than the creating each
experiment so specifically.

In [ ]:
import experiments
from qelement_qops_helper import *

In [ ]:
# Defining QuantumElement with their associated QuantumParameters class

# C2PHI
@attrs.define(kw_only=True)
class C2PhiParameters(QuantumParameters):
    """C2PhiParams parameters.

    Attributes
    ----------
    resonance_frequency_ge:
        The resonance frequency of the 0-1 transition (Hz).
    drive_lo_frequency:
        The frequency of the drive signal local oscillator (Hz).
    """

    resonance_frequency_ge: float | None = None
    drive_lo_frequency: float | None = None
    readout_resonator_frequency: float | None
    readout_lo_frequency: float | None = None 
    resonance_frequency_ge: float | None = None
    drive_lo_frequency: float | None = None
    readout_integration_delay: float | None = None
    readout_range_out: float | None = None
    drive_range: float | None = None
    readout_range_in: float | None = None
    pulse_length: float | None = None
    readout_len: float | None = None
    time_domain_reset_length: float | None = None
    cw_reset_length: float | None = None
    readout_amp: float | None = None
    amplitude_pi: float | None = None
    amplitude_pi_div_2: float | None = None
    

@attrs.define()
class C2Phi(QuantumElement):
    PARAMETERS_TYPE = C2PhiParameters
    
    REQUIRED_SIGNALS = (
        "acquire",
        "drive",
        "measure",
    )
    
    OPTIONAL_SIGNALS = (
        "fast_flux",
        "slow_flux",
    )
    SIGNAL_ALIASES = {}
   
    def calibration(self) -> Calibration:
        """Calibration for the C2Phi"""
        # define the local oscillator if `drive_lo_frequency` was specified:
        if self.parameters.drive_lo_frequency is not None:
            drive_lo = Oscillator(
                uid=f"{self.uid}_drive_local_osc",
                frequency=self.parameters.drive_lo_frequency,
            )
        else:
            drive_lo = None
        # calculate the drive line RF frequency:
        if (
            self.parameters.drive_lo_frequency is not None
            and self.parameters.resonance_frequency_ge is not None
        ):
            drive_rf_frequency = (
                self.parameters.resonance_frequency_ge
                - self.parameters.drive_lo_frequency
            )
        else:
            drive_rf_frequency = None
        calibration = {}
        # define the drive signal calibration:
        sig_cal = SignalCalibration()
        if drive_rf_frequency is not None:
            sig_cal.oscillator = Oscillator(
                uid=f"{self.uid}_drive_ge_osc",
                frequency=drive_rf_frequency,
                modulation_type=ModulationType.AUTO,
            )
        sig_cal.local_oscillator = drive_lo
        calibration[self.signals["drive"]] = sig_cal
        return Calibration(calibration)

###############################################################################

# Fluxonium
@attrs.define(kw_only=True)
class FluxoniumParameters(QuantumParameters):
    """Fluxonium parameters.

    Attributes
    ----------
    resonance_frequency_ge:
        The resonance frequency of the 0-1 transition (Hz).
    drive_lo_frequency:
        The frequency of the drive signal local oscillator (Hz).
    readout_len:
        The length that the readout pulse is played (s).
    readout_amp:
        The amplitude of the readout pulse (arb).
    """

    resonance_frequency_ge: float | None = None
    drive_lo_frequency: float | None = None
    readout_resonator_frequency: float | None
    readout_lo_frequency: float | None = None 
    resonance_frequency_ge: float | None = None
    drive_lo_frequency: float | None = None
    readout_integration_delay: float | None = None
    readout_range_out: float | None = None
    drive_range: float | None = None
    readout_range_in: float | None = None
    pulse_length: float | None = None
    readout_len: float | None = None
    time_domain_reset_length: float | None = None
    cw_reset_length: float | None = None
    readout_amp: float | None = None
    amplitude_pi: float | None = None
    amplitude_pi_div_2: float | None = None

@attrs.define()
class Fluxonium(QuantumElement):
    PARAMETERS_TYPE = FluxoniumParameters
    
    REQUIRED_SIGNALS = (
        "acquire",
        "drive",
        "measure",
    )
    
    OPTIONAL_SIGNALS = (
        "fast_flux",
        "slow_flux"
    )
    SIGNAL_ALIASES = {}

    def calibration(self) -> Calibration:
        """Calibration for the Fluxonium"""
        # define the local oscillator if `drive_lo_frequency` was specified:
        if self.parameters.drive_lo_frequency is not None:
            drive_lo = Oscillator(
                uid=f"{self.uid}_drive_local_osc",
                frequency=self.parameters.drive_lo_frequency,
            )
        else:
            drive_lo = None
        # calculate the drive line RF frequency:
        if (
            self.parameters.drive_lo_frequency is not None
            and self.parameters.resonance_frequency_ge is not None
        ):
            drive_rf_frequency = (
                self.parameters.resonance_frequency_ge
                - self.parameters.drive_lo_frequency
            )
        else:
            drive_rf_frequency = None
        calibration = {}
        # define the drive signal calibration:
        sig_cal = SignalCalibration()
        if drive_rf_frequency is not None:
            sig_cal.oscillator = Oscillator(
                uid=f"{self.uid}_drive_ge_osc",
                frequency=drive_rf_frequency,
                modulation_type=ModulationType.AUTO,
            )
        sig_cal.local_oscillator = drive_lo
        # calibration[self.signals["drive"]] = sig_cal
        return Calibration(calibration)

In [ ]:
F1 = Fluxonium(
    uid='F1',
    signals={
        'acquire': 'F1/acquire_line', # These are just dictionary keys for the path to the logical signal group as defined in the YAML
        'drive': 'F1/drive_line',
        'measure': 'F1/measure_line',
    },
    parameters={
        'resonance_frequency_ge': 1,
        'readout_resonator_frequency':6866759709,
        'readout_lo_frequency':6.8e9,
        'resonance_frequency_ge':386e6,
        'drive_lo_frequency':3.6e9,
        'readout_integration_delay':88e-9,
        'readout_range_out':-20,
        'drive_range':10,
        'readout_range_in':10,
        'pulse_length': 500e-9,
        'readout_len': 2e-6,
        'time_domain_reset_length': 300e-6,
        'cw_reset_length': 5e-9,
        'readout_amp': 0.5,
        'amplitude_pi': 0.32,
        'amplitude_pi_div_2': 0.18,
    }
)

# Load in text file defining logical signal layout
with open('../device_description/Cos2phi_v2_separated_style.txt') as f:
    descriptor_shfqc = f.read()

# Define and Load our Device Setup
device_setup = DeviceSetup.from_descriptor(
    yaml_text=descriptor_shfqc, # yaml fully describes logical signal layout
    server_host="localhost",  # LabOne dataserver host name
    server_port="8004",  # port number of the dataserver - default is 8004
    setup_name="ersevim_host",  # setup name
)

qubit = F1
device_setup.set_calibration(qubit.calibration()) #Uses qubit params for calib
# qpu = QPU(qubits=[qubit], quantum_operations=CustomGeneralOperations) # Does work right now but is overkill
# qt_platform = QuantumPlatform(setup=device_setup, qpu=qpu)

lsg = {
    qubit_name: device_setup.logical_signal_groups[qubit_name].logical_signals
    for qubit_name in device_setup.logical_signal_groups.keys()}

type(lsg['T0']['acquire_line'])

In [ ]:
ro_frequency_sweep = LinearSweepParameter(
    uid='Readout_Frequency_Sweep',
    start=qubit.parameters.readout_resonator_frequency-qubit.parameters.readout_lo_frequency + 10e6,
    stop=qubit.parameters.readout_resonator_frequency-qubit.parameters.readout_lo_frequency - 10e6, 
    count=201)

exp_signals = [
    ExperimentSignal(uid=qubit.signals['measure'], map_to=lsg[qubit.uid]['measure_line']),
    ExperimentSignal(uid=qubit.signals['acquire'], map_to=lsg[qubit.uid]['acquire_line']),
    ExperimentSignal(uid=qubit.signals['drive'], map_to=lsg[qubit.uid]['drive_line'])]

exp = Experiment(
    uid="Resonator Trace",
    signals=exp_signals,)

with exp.acquire_loop_rt(
    uid='shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.SPECTROSCOPY,
):
    with exp.sweep(
        uid='Readout Frequency Sweep',
        parameter=ro_frequency_sweep,
        reset_oscillator_phase=False
    ):
        exp.add(qops.measure(F1, 'single_freq_data'))
            
        # with exp.section(uid="Pulsed Single Frequency Readout"):
        #     exp.play(
        #         signal="measure",
        #         pulse=readout_pulse
        #     )
        #     exp.acquire( #acquire on its own seems to be able to measure things, but not as well(?)
        #         signal='acquire',
        #         handle='single_freq_data',
        #         length=3e-6
        #     )
        with exp.section(uid="delay_between_readout", length=qubit.parameters.cw_reset_length):
            exp.reserve(signal=qubit.signals['measure'])
            exp.reserve(signal=qubit.signals['acquire'])

exp_calibration = Calibration()
readout_osc = Oscillator(
    "readout_osc",
    frequency=ro_frequency_sweep,
    modulation_type=ModulationType.HARDWARE,)
drive_osc = Oscillator(
    "drive_osc",
    frequency=0,
    modulation_type=ModulationType.HARDWARE,)
exp_calibration[qubit.signals['measure']] = SignalCalibration(
    oscillator=readout_osc,)
exp_calibration[qubit.signals['acquire']] = SignalCalibration(
    oscillator=readout_osc,
    port_delay=qubit.parameters.readout_integration_delay)
exp_calibration[qubit.signals['drive']] = SignalCalibration(
    local_oscillator=drive_osc,
    port_mode=PortMode.LF)
exp.set_calibration(exp_calibration)

session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect();
compiled_session = session.compile(exp);

print(exp)

In [ ]:
#--- Running Session ---
results = session.run()
my_results = session.get_results() #a deep copy of session.results

# Extracts data from the exp.acquire method with the same key name
my_acquired_results = my_results.acquired_results['single_freq_data']

In [ ]:
def adjust_phase(
        IQ_data: np.ndarray, frequency: np.ndarray, 
        electrical_delay: float,
        ) -> np.ndarray:
    adjusted_complex = np.exp(1j*electrical_delay*2*np.pi*frequency)*IQ_data
    flattened_angle = np.unwrap(np.angle(adjusted_complex))
    flattened_angle = flattened_angle - np.mean(flattened_angle)
    return flattened_angle 

In [ ]:
results = session.run()
my_results = session.get_results() #a deep copy of session.results
my_acquired_results = my_results.acquired_results['single_freq_data'] #extracts data from the exp.acquire method with the same key name

# For plotting current vs single resonator point
freqs = my_acquired_results.axis[0] + qubit.parameters.readout_lo_frequency
IQ_data = my_acquired_results.data
amplitude = np.abs(IQ_data)
phase = adjust_phase(IQ_data, freqs, qubit.parameters.readout_integration_delay)

# Plots
fig, ax = plt.subplots(2,1, figsize=(8,6))
ax[0].plot(freqs, amplitude)
ax[0].set_title(f'{qubit.uid} Near Resonator Pulsed Trace')
ax[0].set_xlabel('Frequency (GHz)')
ax[0].set_ylabel('Amplitude (a.u.)')
ax[0].grid()
ax[1].plot(freqs, phase)
ax[1].set_title(f'{qubit.uid} Near Resonator Pulsed Trace')
ax[1].set_xlabel('Frequency (GHz)')
ax[1].set_ylabel('Phase')
ax[1].grid()
fig.tight_layout()
# ax[0].vlines(qubit.res_to_current(coil.current()),np.min(amplitude), np.max(amplitude), color='r')
# ax[0].vlines(qubit.parameters.readout_resonator_frequency, np.min(amplitude), np.max(amplitude), colors='r');
# ax[1].vlines(qubit.parameters.readout_resonator_frequency, np.min(phase), np.max(phase), colors='r');

### OLD CODE

In [ ]:
testops = TunableTransmonOperations()
testops.keys()
testops.measure.src

In [ ]:
@dsl.qubit_experiment
def my_exp(qops: dsl.QuantumOperations = qops, q: QuantumElement = qubit):
    dsl.add(qops.measure(q, 'my_handle'))
test = my_exp(qops, qubit)


@dsl.qubit_experiment
def local_resonator_trace(q: QuantumElement, qops: dsl.QuantumOperations):
    '''my docstring'''
    with dsl.acquire_loop_rt(count=100):
        dsl.add(qops.measure(q, 'my_handle'))


# @dsl.experiment(
# def local_resonator_trace(q: QuantumElement, qops: dsl.QuantumOperations):
    # '''my docstring'''
    # dsl.add(qops.measure(q, 'my_handle'))



In [ ]:
exp = local_resonator_trace(qubit, qops)
# print(exp)

session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect();
compiled_session = session.compile(exp);

# print(qubit)
# print(device_setup)

In [ ]:



# qubit.signals['measure']

qops = CustomGeneralOperations()
qops.keys()
qops.measure(F1, 'my_handle')
# qops.rabi?
# qops.rabi.src

In [ ]:
# Defining QuantumElement with their associated QuantumParameters class

# C2PHI
@attrs.define(kw_only=True)
class C2PhiParameters(TunableTransmonQubitParameters):
    pass
    
@attrs.define()
class C2Phi(TunableTransmonQubit):
    PARAMETERS_TYPE = C2PhiParameters

###############################################################################

# Fluxonium
@attrs.define(kw_only=True)
class FluxoniumParameters(TunableTransmonQubitParameters):
    pass

@attrs.define()
class Fluxonium(TunableTransmonQubit):
    PARAMETERS_TYPE = FluxoniumParameters

In [ ]:
qubit = F1
averages=2**8

readout_pulse = pulse_library.gaussian_square(
    uid=f"readout_pulse_{qubit.uid}",
    length=qubit.parameters.readout_len,
    amplitude=qubit.parameters.readout_amp,
    width=qubit.parameters.readout_len*0.9,
    sigma=0.2,)

readout_lo_freq_sweep = LinearSweepParameter(
    uid='Readout_LO',
    start=6.2e9,
    stop=7.2e9,
    count=2)

readout_freq_sweep = LinearSweepParameter(
    uid='Readout_Frequency',
    start=-500e6,
    stop=499e6,
    count=500)

exp_signals = [
    ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
    ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),]

exp = Experiment(
    uid='Global Resonator Trace',
    signals=exp_signals,)
RO_LO_Sweep = Sweep(
    uid='Readout LO Frequency Sweep',
    parameters=readout_lo_freq_sweep)
RT_Loop = AcquireLoopRt(
    uid='Shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.SPECTROSCOPY,)
AWG_Freq_Sweep = Sweep(
    uid='Readout Frequency Sweep',
    parameters=readout_freq_sweep,
    reset_oscillator_phase=False)
Meas_Acquire = Section(uid='Pulsed Single Frequency Readout')
Meas_Acquire.play(
    signal='measure',
    pulse=readout_pulse)
Meas_Acquire.acquire(
    signal='acquire', 
    handle='single_freq_data', 
    length=qubit.parameters.readout_len)
Delay_After_Count = Section(uid='Delay Between Readout')

#--- Properly Defining Nesting Order ---
exp.add(RO_LO_Sweep)
RO_LO_Sweep.add(RT_Loop)
RT_Loop.add(AWG_Freq_Sweep)
AWG_Freq_Sweep.add(Meas_Acquire)
AWG_Freq_Sweep.add(Delay_After_Count)
Delay_After_Count.reserve(signal='measure')
Delay_After_Count.reserve(signal='acquire')

#--- Defines Oscillators ---
readout_osc = Oscillator(
    "readout_osc",
    frequency=readout_freq_sweep,
    modulation_type=ModulationType.HARDWARE)
readout_lo = Oscillator(
    "readout_lo",
    frequency=readout_lo_freq_sweep,
    modulation_type=ModulationType.HARDWARE)

#---Calibration object Updates and Application ---
exp_calibration = Calibration()
exp_calibration["measure"] = SignalCalibration(
    oscillator=readout_osc,
    local_oscillator=readout_lo)
exp_calibration["acquire"] = SignalCalibration(
    oscillator=readout_osc,
    local_oscillator=readout_lo)
exp.set_calibration(exp_calibration)

#--- Compiling Session ---
session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,)
session.connect(use_async_api=True);
compiled_session = session.compile(exp);
# psv.interactive_psv(compiled_session, max_events_to_publish=1000,)

#--- Running Session ---
results = session.run()
my_results = session.get_results() #a deep copy of session.results

# Extracts data from the exp.acquire method with the same key name
my_acquired_results = my_results.acquired_results['single_freq_data']